In [1]:
from pathlib import Path
import os,ast,json,hashlib
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
import numpy as np,h5py,torch
from torch import nn
from sklearn.metrics import roc_auc_score,average_precision_score
ROOT=Path(r'C:/Users/choud/Desktop/Minor project');RUN=ROOT/'fusion/cnn_bilstm_v1';OUT=RUN/'codex_independent_audit';V3=ROOT/'cnn/cnn wave/eew_causal_v3';CR=V3/'cnn_eew_training';BR=ROOT/'bilstm_eew/bilstm_training_v1';TAB=ROOT/'bilstm_eew/causal_tabular_v1/tabular_features.h5'
DEV=torch.device('cuda' if torch.cuda.is_available() else 'cpu');torch.set_num_threads(4);torch.use_deterministic_algorithms(True);torch.backends.cudnn.benchmark=False
def sha(p):
 h=hashlib.sha256()
 with p.open('rb') as f:
  for b in iter(lambda:f.read(4194304),b''):h.update(b)
 return h.hexdigest()
cks={h:torch.load(RUN/'models'/f'fusion_{h}s_best.pt',map_location='cpu',weights_only=False) for h in [1,2,3]}
paths=[RUN/'final_fusion_report.md',RUN/'01_fusion_training.ipynb',RUN/'02_fusion_evaluation.ipynb',TAB,CR/'01_cnn_training.ipynb',BR/'01_bilstm_training.ipynb',CR/'codex_performance_audit/cnn_performance_audit_results.json',BR/'codex_independent_audit/audit_results.json']+[RUN/'models'/f'fusion_{h}s_best.pt' for h in [1,2,3]]+[Path(cks[h]['backbones'][k]) for h in [1,2,3] for k in ['cnn','bilstm']]+[V3/'dataset'/f'eew_dataset_{s}.h5' for s in ['train','validation','test']]
before={str(p):sha(p) for p in paths};report=(RUN/'final_fusion_report.md').read_text(encoding='utf-8');reported=json.loads(report.split('```json\n')[1].split('\n```')[0]);cref=json.loads((CR/'codex_performance_audit/cnn_performance_audit_results.json').read_text(encoding='utf-8'))['common_cohort'];bref=json.loads((BR/'codex_independent_audit/audit_results.json').read_text(encoding='utf-8'))['horizons']
R={'checks':{},'horizons':{},'input_hashes':before}
def check(k,ok,evidence):R['checks'][k]={'status':'PASS' if ok else 'FAIL','evidence':evidence}
# Only exact architecture declarations imported; audit independently implements embeddings, pairing, inference and metrics.
for path,cell,name in [(CR/'01_cnn_training.ipynb',1,'EarlyCNN'),(BR/'01_bilstm_training.ipynb',0,'TabularBiLSTM')]:
 tree=ast.parse(''.join(json.loads(path.read_text(encoding='utf-8'))['cells'][cell]['source']));node=next(n for n in tree.body if isinstance(n,ast.ClassDef) and n.name==name);exec(compile(ast.Module(body=[node],type_ignores=[]),'class_definition_only','exec'))
class Head(nn.Module):
 def __init__(self):
  super().__init__();self.shared=nn.Sequential(nn.Linear(64,32),nn.SiLU(),nn.Dropout(.2));self.detection=nn.Linear(32,1);self.magnitude=nn.Linear(32,1)
 def forward(self,x):
  x=self.shared(x);return self.detection(x)[:,0],self.magnitude(x)[:,0]
def detection(y,p,t=.5):
 y=np.asarray(y)==1;q=p>=t;tp=int((y&q).sum());tn=int((~y&~q).sum());fp=int((~y&q).sum());fn=int((y&~q).sum())
 return dict(accuracy=(tp+tn)/len(y),precision=tp/(tp+fp) if tp+fp else 0.,recall=tp/(tp+fn),f1=2*tp/(2*tp+fp+fn),false_positive_rate=fp/(fp+tn),roc_auc=float(roc_auc_score(y,p)),pr_auc_ap=float(average_precision_score(y,p)),confusion_matrix=[[tn,fp],[fn,tp]])
def magnitude(a,p):
 a=a.astype(float);p=p.astype(float);e=p-a;s=a>=5
 return dict(mae=float(abs(e).mean()),rmse=float(np.sqrt((e**2).mean())),r2=float(1-(e**2).sum()/((a-a.mean())**2).sum()),bias=float(e.mean()),actual_variance=float(a.var()),predicted_variance=float(p.var()),std_ratio=float(p.std()/a.std()),slope=float(np.polyfit(a,p,1)[0]),strong_n=int(s.sum()),strong_underprediction_percent=float((e[s]<0).mean()*100),strong_bias=float(e[s].mean()))
# Independent parent-key join and metadata checks; no regeneration/full-array causality audit.
MAP={};parents={}
with h5py.File(TAB,'r') as tf:
 for split in ['train','validation','test']:
  with h5py.File(V3/'dataset'/f'eew_dataset_{split}.h5','r') as wf:
   w=wf['parent_id'].asstr()[:];t=tf[split]['parent_id'].asstr()[:];assert len(set(w))==len(w) and len(set(t))==len(t) and set(w)==set(t),'Critical parent join failure'
   lookup={p:i for i,p in enumerate(t)};ix=np.array([lookup[p] for p in w]);MAP[split]=ix;parents[split]=set(w);mismatches=[]
   for key in ['source_id','parent_id','pair_id','combined_index','labels','magnitude','magnitude_mask']+[f'{k}_{h}s' for h in [1,2,3] for k in ['classification_mask','magnitude_mask','pre_s_valid']]:
    a=wf[key].asstr()[:] if wf[key].dtype.kind=='O' else wf[key][:];ds=tf[split][key];b=ds.asstr()[:] if ds.dtype.kind=='O' else ds[:]
    if not np.array_equal(a,b[ix]):mismatches.append(key)
   check(split+'_pairing',not mismatches,{'records':len(w),'mismatches':mismatches,'permutation_nonidentity_rows':int((ix!=np.arange(len(w))).sum())})
   for h in [1,2,3]:
    y=wf['labels'][:];reg=wf[f'magnitude_mask_{h}s'][:];keep=wf[f'classification_mask_{h}s'][:];pre=(y==1)&(wf['reference_sample'][:]+h*100<wf['s_arrival_sample'][:])
    check(f'{split}_{h}s_masks_cutoff',np.array_equal(reg,pre) and np.array_equal(keep,pre|(y==0)) and np.all(wf['magnitude'][:][y==0]==-1) and wf[f'windows_{h}s_norm'].shape[1]==100+h*100 and tf[split][f'{h}s/X'].shape[1:]==(10+h*10,7),'Labels/first-S eligibility/noise sentinel; only observed horizon enters models.')
    sg=tf[f'scalers/{h}s'];check(f'{split}_{h}s_feature_scaling_metadata',sg.attrs['fit_split']=='train' and sg.attrs['fit_records']==int(tf[f'train/classification_mask_{h}s'][:].sum()),'Approved precomputed feature scaler provenance reused; feature extraction not rerun.')
for a,b in [('train','validation'),('train','test'),('validation','test')]:check(a+'/'+b+'_parent_isolation',not(parents[a]&parents[b]),len(parents[a]&parents[b]))
@torch.inference_mode()
def encode(split,h,c,b,cc,bc,common=False):
 with h5py.File(V3/'dataset'/f'eew_dataset_{split}.h5','r') as wf,h5py.File(TAB,'r') as tf:
  mask=wf[f'classification_mask_{h}s'][:].astype(bool)
  if common:mask=np.logical_and.reduce([wf[f'classification_mask_{j}s'][:] for j in [1,2,3]])
  x=wf[f'windows_{h}s_norm'][:][mask,None,:];tab=tf[split][f'{h}s/X'][:][MAP[split]][mask];y=wf['labels'][:][mask];m=wf['magnitude'][:][mask];reg=wf[f'magnitude_mask_{h}s'][:][mask];ids=wf['source_id'].asstr()[:][mask];pid=wf['parent_id'].asstr()[:][mask]
 em=[];cp=[];cm=[];bp=[];bm=[]
 for i in range(0,len(x),256):
  xx=torch.from_numpy(x[i:i+256]).to(DEV);tt=torch.from_numpy(tab[i:i+256]).to(DEV)
  u=c.features(xx.sign()*xx.abs().log1p());ce=c.shared(torch.cat((u.mean(-1),u.amax(-1)),1));v,_=b.lstm(tt);be=b.shared(v.mean(1));em.append(torch.cat((ce,be),1).cpu())
  cp.append(c.detection(ce).flatten().sigmoid().cpu());cm.append(c.magnitude(ce).flatten().cpu());bp.append(b.detect(be).flatten().sigmoid().cpu());bm.append(b.mag(be).flatten().cpu())
 return dict(z=torch.cat(em),y=y,m=m,reg=reg,ids=ids,parent=pid,cp=torch.cat(cp).numpy(),cm=torch.cat(cm).numpy()*cc['target_scale']+cc['target_center'],bp=torch.cat(bp).numpy(),bm=torch.cat(bm).numpy()*bc['scale']+bc['center'])
@torch.inference_mode()
def infer(head,z,mu,sd):
 logits=[];pred=[]
 for chunk in ((z-mu)/sd).split(256):
  a,b=head(chunk.to(DEV));logits.append(a.cpu());pred.append(b.cpu())
 return torch.cat(logits),torch.cat(pred)
def uncertainty(y,target,fp,fm,bp,bm):
 rng=np.random.default_rng(2026);eq=np.flatnonzero(y==1);noise=np.flatnonzero(y==0);df=[];dm=[]
 def f1(y,p):
  a=y==1;b=p>=.5;tp=(a&b).sum();return 2*tp/(2*tp+(~a&b).sum()+(a&~b).sum())
 for _ in range(500):
  e=rng.choice(eq,len(eq),replace=True);n=rng.choice(noise,len(noise),replace=True);ix=np.r_[e,n];df.append(float(f1(y[ix],fp[ix])-f1(y[ix],bp[ix])));dm.append(float((abs(fm[e]-target[e])-abs(bm[e]-target[e])).mean()))
 return {'f1_difference_ci95':np.quantile(df,[.025,.975]).tolist(),'mae_difference_ci95':np.quantile(dm,[.025,.975]).tolist()}
for h,ck in cks.items():
 cc=torch.load(ck['backbones']['cnn'],map_location='cpu',weights_only=False);bc=torch.load(ck['backbones']['bilstm'],map_location='cpu',weights_only=False)
 c=EarlyCNN().to(DEV);b=TabularBiLSTM().to(DEV);c.load_state_dict(cc['state_dict'],strict=True);b.load_state_dict(bc['state_dict'],strict=True)
 for model in [c,b]:model.requires_grad_(False);model.eval()
 head=Head().to(DEV);head.load_state_dict(ck['head_state'],strict=True);head.eval()
 check(f'{h}s_backbone_checkpoint_preservation',all(before[ck['backbones'][k]]==ck['input_hashes'][ck['backbones'][k]] for k in ['cnn','bilstm']) and all(not p.requires_grad for model in [c,b] for p in model.parameters()) and not any(k.startswith(('lstm','features')) for k in ck['head_state']),{'source_hashes_match_training_snapshot':True,'fusion_checkpoint':'head-only weights; original backbone files used','scope':'Saved training source uses inference-mode cached embeddings, eval frozen backbones, head-only optimizer and tensor-exact end checks. No historical optimizer trace exists beyond these artifacts.'})
 tr=encode('train',h,c,b,cc,bc);mu=tr['z'].mean(0);sd=tr['z'].std(0,unbiased=False);sd=torch.where(sd>1e-8,sd,torch.ones_like(sd));center=float(torch.from_numpy(tr['m'][tr['reg']]).mean());scale=float(torch.from_numpy(tr['m'][tr['reg']]).std(unbiased=False))
 check(f'{h}s_train_only_fusion_scalers',torch.equal(mu,ck['embedding_mean']) and torch.equal(sd,ck['embedding_scale']) and center==ck['target_center'] and scale==ck['target_scale'],{'embedding_mean_max_difference':float((mu-ck['embedding_mean']).abs().max()),'embedding_scale_max_difference':float((sd-ck['embedding_scale']).abs().max()),'target_center':center,'target_scale':scale})
 va=encode('validation',h,c,b,cc,bc);vl,vz=infer(head,va['z'],mu,sd);vp=vl.sigmoid().numpy();vr=va['reg'];loss=float(nn.functional.binary_cross_entropy_with_logits(vl,torch.tensor(va['y'],dtype=torch.float32)))+float(nn.functional.huber_loss(vz[vr],(torch.tensor(va['m'])[vr]-center)/scale,delta=1.))
 best=float('inf');ep=None;stale=0
 for row in ck['history']:
  if row['validation']['total']<best-ck['config']['min_delta']:best=row['validation']['total'];ep=row['epoch'];stale=0
  else:stale+=1
 grid=np.round(np.arange(.10,.901,.01),2);scores=[detection(va['y'],vp,t)['f1'] for t in grid];j=max(range(len(grid)),key=lambda i:(scores[i],-abs(grid[i]-.5),-grid[i]));threshold=float(grid[j]);noise=vp[va['y']==0];allowed=int(np.floor(.05*len(noise)));low=float(np.nextafter(np.sort(noise)[-(allowed+1)],np.float32(np.inf)))
 check(f'{h}s_validation_epoch_thresholds',ep==ck['best_epoch']=={1:2,2:15,3:27}[h] and abs(loss-ck['best_validation_loss'])<1e-6 and threshold==ck['threshold'] and low==ck['low_fpr_threshold'] and stale==7,{'epoch':ep,'epochs_run':len(ck['history']),'validation_loss_delta':loss-ck['best_validation_loss'],'threshold':threshold,'low_fpr_threshold':low,'stale_epochs':stale})
 if h==1:R['epoch2_review']={'status':'PASS_WITH_LIMITATION','history':ck['history'],'finding':'Epoch2 is selected by recorded validation loss and reproduced frozen loss; training stopped after seven non-improving epochs at epoch9. Early selection alone is not an error. Limited single-seed head adaptation/validation fluctuations do not establish optimal convergence.'}
 d=encode('test',h,c,b,cc,bc,True);l,z=infer(head,d['z'],mu,sd);p=l.sigmoid().numpy();m=z.numpy()*scale+center;reg=d['reg'];y=d['y'];target=d['m'];digest=hashlib.sha256('\n'.join(d['ids']).encode()).hexdigest()
 check(f'{h}s_common_cohort',len(y)==2414 and int(reg.sum())==1200 and int((~reg).sum())==1214 and digest==cref['identity_sha256'],{'source_order_sha256':digest,'parent_order_sha256':hashlib.sha256('\n'.join(d['parent']).encode()).hexdigest(),'earthquakes':int(reg.sum()),'noise':int((~reg).sum())})
 values={'fixed_05':detection(y,p,.5),'validation_selected':detection(y,p,threshold),'validation_low_fpr':detection(y,p,low),'magnitude':magnitude(target[reg],m[reg])};diffs={}
 for key,ref in [('fixed_05',reported[str(h)]['models']['Fusion']['fixed_05']),('validation_selected',reported[str(h)]['fusion_selected']),('validation_low_fpr',reported[str(h)]['fusion_low_fpr_test']),('magnitude',reported[str(h)]['models']['Fusion']['magnitude'])]:
  for k,v in values[key].items():
   if k=='confusion_matrix':check(f'{h}s_{key}_counts',v==ref[k],v)
   else:diffs[key+'/'+k]=v-ref[k]
 check(f'{h}s_fusion_metrics',max(abs(v) for v in diffs.values())<1e-6,{'max_difference':max(abs(v) for v in diffs.values()),'tolerance':1e-6})
 paired={};baselines={}
 for name,prob,pred,ref in [('CNN',d['cp'],d['cm'],cref['metrics'][str(h)]),('BiLSTM',d['bp'],d['bm'],bref[str(h)]['common']['fixed_05_detection'])]:
  dm=detection(y,prob,.5);mm=magnitude(target[reg],pred[reg]);baselines[name]={'detection':dm,'magnitude':mm};check(f'{h}s_{name}_policy',all(abs(dm[k]-ref[k])<1e-6 for k in ['accuracy','precision','recall','f1']),'Identical parent-keyed records; fixed .5 for all three models.')
  q=uncertainty(y,target,p,m,prob,pred);old=reported[str(h)]['paired_comparisons'][name];check(f'{h}s_paired_{name}',all(np.allclose(q[k],old[k],rtol=0,atol=1e-6) for k in q),q);paired[name]={'f1_delta':values['fixed_05']['f1']-dm['f1'],'mae_delta':values['magnitude']['mae']-mm['mae'],**q}
 R['horizons'][h]={'epoch':ep,'threshold':threshold,'low_fpr_threshold':low,**values,'reported_differences':diffs,'baselines':baselines,'paired':paired}
 print(f'{h}s verified: epoch{ep}, F1={values["fixed_05"]["f1"]:.6f}, MAE={values["magnitude"]["mae"]:.6f}')
check('saved_protocol_no_test_tuning',True,'Reviewed training/evaluation cells: only training embeddings/scalers and validation losses/F1/negative-score quantiles select heads/thresholds; all three saved before test inference. Test bootstrap is descriptive only. No new threshold or training performed by audit.')
R['checks']['historical_execution_certainty']={'status':'UNKNOWN','evidence':'Saved code, history, source hashes, scaler replay and frozen loss reproduction support protocol; no immutable independent log of every past action. Previously examined test cohort is exploratory, not untouched confirmatory evidence.'}
R['checks']['noise_family_shortcuts']={'status':'UNKNOWN','evidence':'No new family audit conducted; both branches share synthetic observations, so fusion does not establish independent information or shortcut removal.'}
check('preservation',all(sha(Path(p))==v for p,v in before.items()),{'original_files_hashed':len(before),'changed':[]})
R['verdict']='BLOCKED' if any(c['status']=='FAIL' for c in R['checks'].values()) else 'APPROVED'
R['scope']='Valid synthetic oracle-aligned fusion benchmark reporting and controlled comparison, not operational EEW or a claim of improved performance. Attention not implemented or started.'
with (OUT/'audit_results.json').open('x',encoding='utf-8') as f:json.dump(R,f,indent=2,allow_nan=False)
lines=['# Independent CNN + BiLSTM fusion audit','', '**Date:** 2026-11-13. **Verdict: '+R['verdict']+'**','',R['scope'],'','## Check results','','| Check | Status | Evidence |','|---|---|---|']
for k,c in R['checks'].items():lines.append('| '+k+' | '+c['status']+' | '+str(c['evidence']).replace('|','/')+' |')
lines+=['','## Frozen inference reproduction','','Reused only backbone class declarations, independently implemented direct penultimate embedding extraction (without original forward hooks), parent-key joins, train-embedding scaler replay and frozen head inference. No original training/evaluation cells executed. Full-precision signed metric discrepancies are retained in JSON, including confusion matrices and strong-event bias. No feature generation or full dataset audit repeated.','', '| Horizon | Threshold policy | Precision | Recall | F1 | FPR | Accuracy | ROC-AUC | PR-AUC(AP) |','|---|---|---:|---:|---:|---:|---:|---:|---:|']
for h,v in R['horizons'].items():
 for key,label in [('fixed_05','0.5'),('validation_selected',str(v['threshold'])),('validation_low_fpr','low-FPR '+str(v['low_fpr_threshold']))]:
  d=v[key];lines.append(f'| {h}s | {label} | '+' | '.join(f'{d[k]:.8f}' for k in ['precision','recall','f1','false_positive_rate','accuracy','roc_auc','pr_auc_ap'])+' |')
lines+=['','| Horizon | MAE | RMSE | R2 | Bias | Pred/actual SD | Strong N | M>=5 underprediction % | Strong bias |','|---|---:|---:|---:|---:|---:|---:|---:|---:|']
for h,v in R['horizons'].items():
 m=v['magnitude'];lines.append(f"| {h}s | {m['mae']:.9f} | {m['rmse']:.9f} | {m['r2']:.9f} | {m['bias']:.6f} | {m['std_ratio']:.5f} | {m['strong_n']} | {m['strong_underprediction_percent']:.4f} | {m['strong_bias']:.6f} |")
lines+=['','## Epoch 2 review','','The 1s selected epoch2 was explicitly reviewed: replaying min_delta=0.0001 and patience7 selects epoch2 and ends at epoch9. Reloaded loss agrees within1e-6. Later epochs did not satisfy the recorded validation improvement rule. This is not a masking, checkpoint or selection error. Single-seed short head adaptation does not prove optimal convergence; no retraining or tuning is performed or recommended solely because the selected epoch is early. Full nine-epoch history is in JSON.','','## Paired comparisons and interpretation','','| Horizon | Baseline | Fusion-minus-baseline F1 [95% CI] | Fusion-minus-baseline MAE [95% CI] |','|---|---|---|']
for h,v in R['horizons'].items():
 for name,q in v['paired'].items():lines.append(f"| {h}s | {name} | {q['f1_delta']:.6f} {q['f1_difference_ci95']} | {q['mae_delta']:.6f} {q['mae_difference_ci95']} |")
lines+=['','The original 500-resample paired stratified bootstrap (seed2026) reproduces within numerical precision. Same event/noise draws are applied to both models; magnitude uses the paired earthquake draw. These are descriptive percentile intervals, not multiple-comparison-corrected or multi-seed confirmatory evidence. The 1s CNN F1 interval barely excludes zero and is particularly sensitive to bootstrap Monte Carlo uncertainty; do not overinterpret that boundary.','', 'Fusion has no consistent advantage over CNN: its fixed-.5 F1 is lower at 1s/3s, its small 2s gain has an interval spanning zero, and every fusion-versus-CNN MAE interval spans zero. Magnitude compression and substantial M>=5 underprediction remain. Comparisons of validation-selected fusion F1 against fixed-.5 baselines are not equivalent policies; only the common fixed-.5 table supports that policy-matched comparison. Low-FPR thresholds are selected on validation and do not guarantee a fixed test FPR.','', 'The common cohort matches exactly 1,200 earthquakes and 1,214 noise. All labels, magnitude targets, pre-S/regression masks and parent identities agree between modalities; source/parent metadata never enters embeddings. Fusion embedding and target scalers replay exactly from eligible training data. Approved tabular scaler provenance is reused rather than re-auditing extraction.','', 'Backbone preservation is supported by original-checkpoint hashes matching fusion training snapshots, a head-only fusion checkpoint, cached inference-mode embeddings, head-only optimizer and original source tensor-equality guards. This audit loads unchanged weights; historical intermediate runtime states are not independently logged.','', 'Both branches derive from the same synthetic observations and assume oracle P/reference alignment. Model capacity/representation/budget differences prevent an isolated architecture claim. No real-world EEW readiness, autonomous P picker, operational latency or warning lead time is demonstrated. Previously inspected test data must not be used to tune future Attention. No Attention, cleanup or further experiment began.','','## Evidence hashes','']
for p,v in before.items():lines.append(f'- `{p}` — `{v}`')
lines+=['','Only audit.ipynb, audit_report.md and audit_results.json were created. Existing inputs/checkpoints/notebooks/reports unchanged.']
with (OUT/'audit_report.md').open('x',encoding='utf-8') as f:f.write('\n'.join(lines)+'\n')
print('Verdict',R['verdict'],'failed checks',[k for k,v in R['checks'].items() if v['status']=='FAIL'])


1s verified: epoch2, F1=0.679908, MAE=0.664240
2s verified: epoch15, F1=0.716312, MAE=0.635879
3s verified: epoch27, F1=0.747475, MAE=0.608215
Verdict APPROVED failed checks []
